In [ ]:
import pandas as pd
import os
import glob
import re

# ==============================
# FOLDERS
# ==============================
CLIMATE_DIR = r"Data\combined_climate"
WHEAT_DIR   = r"Data\wheat_data_only"
OUT_DIR     = r"Data\master_combined"

os.makedirs(OUT_DIR, exist_ok=True)

# ==============================
# Helper: extract year from filename
# ==============================
def extract_year(filename):
    match = re.search(r"(19|20)\d{2}", filename)
    return match.group() if match else None

# ==============================
# Build wheat file lookup by year
# ==============================
wheat_files = {}
for wf in glob.glob(os.path.join(WHEAT_DIR, "*.csv")):
    year = extract_year(os.path.basename(wf))
    if year:
        wheat_files[year] = wf

# ==============================
# Process climate files
# ==============================
for climate_file in glob.glob(os.path.join(CLIMATE_DIR, "*.csv")):

    fname = os.path.basename(climate_file)
    year = extract_year(fname)

    if not year:
        print(f"⚠️ Skipping (no year): {fname}")
        continue

    if year not in wheat_files:
        print(f"⚠️ No wheat file for year {year}")
        continue

    print(f"🔗 Processing year {year}")

    # Load data
    climate_df = pd.read_csv(climate_file)
    wheat_df   = pd.read_csv(wheat_files[year])

    # Keep only needed columns from wheat
    wheat_df = wheat_df[["lat", "lon", "H_wheat_dot_hat"]]

    # ==============================
    # EXACT spatial merge
    # ==============================
    merged_df = climate_df.merge(
        wheat_df,
        on=["lat", "lon"],
        how="left",
        validate="many_to_one"   # ensures spatial uniqueness
    )

    # ==============================
    # Save output
    # ==============================
    out_file = os.path.join(
        OUT_DIR,
        fname.replace(".csv", "_with_wheat.csv")
    )

    merged_df.to_csv(out_file, index=False)

print("✅ All climate files updated with wheat data.")
